# Assignment 5 — Option A: "Ask My Resume" RAG Chatbot
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Sadaf Sarbazi  
**Date:** May 2026  
**Option:** A — Resume RAG Chatbot  
**API Path:** Free (HuggingFace Inference API + sentence-transformers)  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Document Loading](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Retrieval Chain](#5-retrieval)
6. [Prompt Engineering](#6-prompting)
7. [Evaluation](#7-evaluation)

---
<a id='1-setup'></a>
## 1. Setup and Imports

In [3]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env')

HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found — check your .env file'
print('HF_TOKEN loaded successfully')

HF_TOKEN loaded successfully


In [4]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from huggingface_hub import InferenceClient

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
EMBED_MODEL = 'all-MiniLM-L6-v2'
DATA_DIR = Path('..') / 'data'

print(f'LLM: {MODEL_ID}')
print(f'Embedding model: {EMBED_MODEL}')
print(f'Data directory: {DATA_DIR.resolve()}')

LLM: Qwen/Qwen2.5-7B-Instruct
Embedding model: all-MiniLM-L6-v2
Data directory: .../bsan6200-assignment5/data


---
<a id='2-loading'></a>
## 2. Document Loading

Loading 4 career documents from `data/`:
- `CV.pdf` — full curriculum vitae
- `Resume.pdf` — professional resume
- `SOQ.pdf` — statement of qualifications / cover letter
- `github_projects.txt` — GitHub portfolio and project summaries

> **Note:** A fifth document, `data/about.txt` (a structured public career summary with dedicated education and skills sections), was added later to improve retrieval quality in the deployed Streamlit app. It is not included in this notebook run but is committed to the repository and loaded automatically by `streamlit_app.py`.

In [6]:
def load_pdf(filepath):
    reader = PdfReader(filepath)
    return '\n'.join(page.extract_text() or '' for page in reader.pages)

def load_txt(filepath):
    return Path(filepath).read_text(encoding='utf-8')

def load_all_documents(data_dir):
    docs = []
    for f in sorted(Path(data_dir).iterdir()):
        if f.suffix == '.pdf':
            text = load_pdf(f)
        elif f.suffix == '.txt':
            text = load_txt(f)
        else:
            continue
        if text.strip():
            docs.append({'text': text, 'source': f.name})
    return docs

documents = load_all_documents(DATA_DIR)

print(f'Documents loaded: {len(documents)}')
for doc in documents:
    print(f"  - {doc['source']}: {len(doc['text'])} characters")
    print(f"    Preview: {doc['text'][:150].strip()}...")
    print()

Documents loaded: 4
  - CV.pdf: 4358 characters
  - Resume.pdf: 5305 characters
  - SOQ.pdf: 9719 characters
  - github_projects.txt: 2196 characters


---
<a id='3-chunking'></a>
## 3. Text Chunking

Comparing two chunking strategies:
- **Strategy 1 — Fixed-size:** Split by character count with overlap
- **Strategy 2 — Paragraph-aware:** Split on double newlines (natural paragraph breaks), then cap long paragraphs

In [8]:
# ── Strategy 1: Fixed-size chunking ──

def chunk_fixed(documents, chunk_size=400, overlap=50):
    chunks = []
    for doc in documents:
        text = doc['text']
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end].strip()
            if chunk:
                chunks.append({'text': chunk, 'source': doc['source']})
            start += chunk_size - overlap
    return chunks

chunks_fixed = chunk_fixed(documents, chunk_size=400, overlap=50)
avg_len_fixed = sum(len(c['text']) for c in chunks_fixed) / len(chunks_fixed)

print(f'Strategy 1 — Fixed-size (size=400, overlap=50)')
print(f'  Total chunks: {len(chunks_fixed)}')
print(f'  Average chunk length: {avg_len_fixed:.0f} characters')
print(f'  Sample chunk:')
print(f"  {chunks_fixed[0]['text'][:200]}...")

Strategy 1 — Fixed-size (size=400, overlap=50)
  Total chunks: 64
  Average chunk length: 377 characters
  Sample chunk:
  Sadaf Sarbazi  M.S. Business Analytics Candidate | Loyola Marymount University
  Profile: Data-driven analytics professional...


In [9]:
# ── Strategy 2: Paragraph-aware chunking ──

def chunk_paragraph(documents, max_chunk=600, overlap=80):
    chunks = []
    for doc in documents:
        paragraphs = re.split(r'\n\s*\n', doc['text'])
        current = ''
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            if len(current) + len(para) + 1 <= max_chunk:
                current = (current + '\n' + para).strip()
            else:
                if current:
                    chunks.append({'text': current, 'source': doc['source']})
                current = (current[-overlap:] + '\n' + para).strip() if current else para
        if current:
            chunks.append({'text': current, 'source': doc['source']})
    return chunks

chunks_para = chunk_paragraph(documents, max_chunk=600, overlap=80)
avg_len_para = sum(len(c['text']) for c in chunks_para) / len(chunks_para)

print(f'Strategy 2 — Paragraph-aware (max=600, overlap=80)')
print(f'  Total chunks: {len(chunks_para)}')
print(f'  Average chunk length: {avg_len_para:.0f} characters')
print(f'  Sample chunk:')
print(f"  {chunks_para[0]['text'][:200]}...")

Strategy 2 — Paragraph-aware (max=600, overlap=80)
  Total chunks: 26
  Average chunk length: 866 characters
  Sample chunk:
  Sadaf Sarbazi  M.S. Business Analytics Candidate | Loyola Marymount University
  Profile: Data-driven analytics professional with experience in ML and data science...


### Chunking Strategy Comparison

| Strategy | Chunk Size | Overlap | Total Chunks | Avg Length |
|----------|-----------|---------|-------------|------------|
| Fixed-size | 400 chars | 50 chars | 64 | 377 chars |
| Paragraph-aware | 600 chars max | 80 chars | 26 | 866 chars |

**Chosen strategy: Paragraph-aware**

**Why:** A resume and SOQ are naturally organized into semantic paragraphs — education, experience, skills, projects. Splitting on paragraph boundaries preserves complete units of meaning, so each retrieved chunk is self-contained and coherent. Fixed-size splitting produces 2.5× more chunks at less than half the average length, frequently cutting mid-sentence and mixing content from different sections, which reduces retrieval precision. The paragraph-aware strategy produces 26 dense, meaningful chunks that align with how the documents are logically structured.

In [11]:
# Use paragraph-aware chunking going forward
final_chunks = chunks_para
print(f'Final chunk count: {len(final_chunks)}')

Final chunk count: 26


---
<a id='4-embedding'></a>
## 4. Embedding and Vector Store

Using `sentence-transformers/all-MiniLM-L6-v2` (free, runs locally) via ChromaDB's embedding function.

> **Note:** This notebook uses ChromaDB for local development. The deployed Streamlit app uses NumPy cosine similarity instead, eliminating the opentelemetry/protobuf dependency that caused import failures on Streamlit Cloud (Python 3.14). Retrieval quality is identical — same embedding model, same cosine distance metric.

### Embedding Model Rationale

**Why `sentence-transformers/all-MiniLM-L6-v2`?**

| Criterion | Decision |
|-----------|----------|
| Task fit | Purpose-built for semantic similarity — maps text to 384-dimensional dense vectors that capture meaning, not just keywords |
| Cost | Runs entirely locally; no API calls or usage fees |
| Speed | Lightweight (22M parameters) — encodes all 26 chunks in under 2 seconds on CPU |
| Benchmarks | Top-performing model on SBERT semantic similarity benchmarks for its size class |
| Retrieval quality | A query like "Python experience" retrieves chunks mentioning "pandas, scikit-learn, Jupyter" because their semantic representations are close — keyword matching would miss this |

**Vector store:** ChromaDB (local development). The deployed Streamlit app replaces ChromaDB with a NumPy dot-product over normalized embeddings — producing identical cosine similarity results with no external dependency.

In [13]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name='resume_rag',
    embedding_function=ef,
)

collection.add(
    documents=[c['text'] for c in final_chunks],
    metadatas=[{'source': c['source']} for c in final_chunks],
    ids=[f'chunk_{i}' for i in range(len(final_chunks))],
)

print(f'Vector store created with {collection.count()} chunks')

Vector store created with 26 chunks


In [14]:
# ── Verify: similarity search ──
test_queries = [
    'Python programming skills',
    'data analytics experience',
    'machine learning projects',
]

for query in test_queries:
    results = collection.query(query_texts=[query], n_results=2)
    print(f'Query: "{query}"')
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print(f"  [{meta['source']}] {doc[:120]}...")
    print()

Query: "Python programming skills"
  [CV.pdf] ...Python, SQL, Jupyter Notebook, Google Colab, statistical modeling...
  [github_projects.txt] ...machine-learning-spring-2026: ML Coursework (BSAN 6070)...

Query: "data analytics experience"
  [CV.pdf] ...M.S. Business Analytics Candidate | Loyola Marymount University...
  [CV.pdf] ...Unsupervised Learning, Clustering, Time-Series Modeling, Sentiment Analysis...

Query: "machine learning projects"
  [github_projects.txt] ...BSAN6070-Final: Paris Agreement Climate Classifier...
  [github_projects.txt] ...Random Forest (selected): 0.933 ROC-AUC, 84.6% accuracy...



---
<a id='5-retrieval'></a>
## 5. Retrieval Chain

Connecting the vector store to `Qwen/Qwen2.5-7B-Instruct` via HuggingFace Inference API (free tier).

In [16]:
hf_client = InferenceClient(token=HF_TOKEN)

def retrieve(query, k=3):
    results = collection.query(query_texts=[query], n_results=k)
    docs = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    return docs, sources

def ask_llm(prompt, max_tokens=300):
    response = hf_client.chat_completion(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return response.choices[0].message.content.strip()

def rag_query(question, system_prompt, k=3):
    docs, sources = retrieve(question, k=k)
    context = '\n\n'.join(docs)
    full_prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    answer = ask_llm(full_prompt)
    return answer, docs, sources

# Sanity check
test_answer, test_docs, test_sources = rag_query(
    'What technical skills does this person have?',
    system_prompt='Answer questions about this person based only on the provided context.'
)
print('Retrieval chain works!')
print(f'Sources used: {test_sources}')
print(f'Answer: {test_answer[:300]}')

Retrieval chain works!
Sources used: ['CV.pdf', 'Resume.pdf', 'Resume.pdf']
Answer: This person has a variety of technical skills, including:
- Data & Programming: Python, SQL, Jupyter Notebook, Google Colab
- Machine Learning: Regression, Supervised & Unsupervised Learning, Clustering,
  Time-Series Modeling, Sentiment Analysis, Exploratory Data Analysis
- Analytics Tools: Power BI, Tableau, statistical analysis frameworks


---
<a id='6-prompting'></a>
## 6. Prompt Engineering

**Note (Tier 2):** Prompt design is your own work. Write 3 iterations below, using the same test question each time to make improvement measurable.

**Consistent test question:** *"What experience does this person have with data analytics?"*

In [1]:
EVAL_QUESTION = 'What experience does this person have with data analytics?'

# ── Iteration 1: Baseline ──
PROMPT_V1 = """You are a chatbot representing Sadaf Sarbazi's professional background. Your job is to answer questions about her experience, skills, education, and projects based on the documents provided."""

answer_v1, _, _ = rag_query(EVAL_QUESTION, PROMPT_V1)
print('=== Prompt v1 Output ===')
print(answer_v1)

=== Prompt v1 Output ===
Sadaf Sarbazi has experience in data analytics through her academic and professional background. She is pursuing an M.S. in Business Analytics at Loyola Marymount University. Her skills include Exploratory Data Analysis, statistical modeling, and tools like Power BI and Tableau. She has also worked with Python and SQL for data analysis tasks and applied machine learning methods in several course projects, including clustering, regression, and time-series modeling.


**v1 → v2: What changed and why?**
> v1 answered accurately but drew on general knowledge alongside the documents — there was no constraint keeping the model within the retrieved context. Adding a grounding constraint in v2 forces the model to stay within the provided context and introduces an explicit fallback phrase for out-of-scope questions.

In [1]:
# ── Iteration 2: Add grounding ──
PROMPT_V2 = """You are a chatbot representing Sadaf Sarbazi's professional background. Your job is to answer questions about her experience, skills, education, and projects based on the documents provided. Only use information from the provided context to answer — do not use outside knowledge. If the context does not contain the answer, say: 'I don't have that information in my documents.'"""

answer_v2, _, _ = rag_query(EVAL_QUESTION, PROMPT_V2)
print('=== Prompt v2 Output ===')
print(answer_v2)

=== Prompt v2 Output ===
Based on the provided documents, Sadaf Sarbazi has data analytics experience through her M.S. Business Analytics program at LMU, where she has studied Exploratory Data Analysis, Unsupervised Learning, Clustering, Time-Series Modeling, and Sentiment Analysis. Her coursework and projects have involved Python, SQL, and tools such as Power BI and Tableau. She has applied these skills in projects including a Paris Agreement climate classifier (BSAN 6070) using Random Forest with 0.933 ROC-AUC, and data visualization work across several other courses.


**v2 → v3: What changed and why?**
> v2 kept answers grounded to the documents and handled out-of-scope questions correctly. v3 adds a recruiter-facing persona and an instruction to use complete sentences, producing cleaner, more professional output appropriate for a hiring context without changing the grounding behavior.

In [1]:
# ── Iteration 3: Final ──
PROMPT_V3 = """You are a professional career chatbot representing Sadaf Sarbazi, designed for recruiters and hiring managers. Answer questions about her experience, skills, education, and projects using only the provided context — do not draw on outside knowledge. If the answer is not in the documents, say: 'I don't have that information in my documents.' Keep responses concise, professional, and in complete sentences."""

answer_v3, _, _ = rag_query(EVAL_QUESTION, PROMPT_V3)
print('=== Prompt v3 Output (Final) ===')
print(answer_v3)

=== Prompt v3 Output (Final) ===
Sadaf Sarbazi has developed strong data analytics capabilities through her M.S. Business Analytics program at Loyola Marymount University. Her technical skills span Python, SQL, Power BI, and Tableau, applied across coursework in Exploratory Data Analysis, Clustering, Time-Series Modeling, and Sentiment Analysis. She has translated these skills into project work, including a machine learning classifier for the Paris Agreement dataset (0.933 ROC-AUC) and a Naive Bayes sentiment analysis pipeline. Her background positions her well for data analyst or business analytics roles where both technical depth and business communication are required.


**Overall improvement from v1 to v3:**
> From v1 to v3, outputs became more grounded, professional, and concise. The grounding constraint added in v2 was the single most impactful change — it eliminated reliance on outside knowledge and introduced a clear out-of-scope fallback. The tone and format instruction added in v3 produced recruiter-appropriate language in complete sentences without changing the grounding behavior. Each iteration produced measurably better output for the same question.

In [1]:
FINAL_PROMPT = PROMPT_V3
print('Final prompt set for evaluation.')

Final prompt set for evaluation.


---
<a id='7-evaluation'></a>
## 7. Evaluation

10 questions across 4 categories. Scoring per question:
- **Retrieval quality:** Yes / Partial / No
- **Faithfulness:** Faithful / Partial / Hallucinated
- **Answer quality:** 1–5

**Note (Tier 2):** Fill in scores and analysis yourself — AI is prohibited from evaluation.

In [1]:
TEST_QUESTIONS = [
    {'q': 'What degree(s) does this person hold?',                          'category': 'Factual'},
    {'q': 'What programming languages does this person know?',              'category': 'Factual'},
    {'q': 'Describe the Paris Agreement climate classifier project.',       'category': 'Factual'},
    {'q': "How does this person's ML experience apply to business problems?", 'category': 'Inference'},
    {'q': 'What type of role would suit this person best and why?',         'category': 'Inference'},
    {'q': "What is this person's salary expectation?",                      'category': 'Out-of-scope'},
    {'q': 'Does this person have 10 years of professional experience?',     'category': 'Out-of-scope'},
    {'q': "What is this person's home address?",                            'category': 'Out-of-scope'},
    {'q': 'What did this person write in their statement of qualifications?','category': 'Specificity'},
    {'q': 'What model did the BSAN6070 final project use and why?',         'category': 'Specificity'},
]

results = []
for item in TEST_QUESTIONS:
    answer, docs, sources = rag_query(item['q'], FINAL_PROMPT, k=3)
    results.append({'category': item['category'], 'question': item['q'],
                    'sources': sources, 'answer': answer})
    print(f"[{item['category']}] {item['q']}")
    print(f"  Sources: {sources}")
    print(f"  Answer: {answer[:200]}")
    print()

[Factual] What degree(s) does this person hold?
  Sources: ['Resume.pdf', 'CV.pdf', 'CV.pdf']
  Answer: I don't have that information in my documents.

[Factual] What programming languages does this person know?
  Sources: ['Resume.pdf', 'github_projects.txt', 'Resume.pdf']
  Answer: Based on the provided documents, this person knows Python and works with Jupyter Notebook and Google Colab. The documents also reference SQL usage in data analytics coursework.

[Factual] Describe the Paris Agreement climate classifier project.
  Sources: ['github_projects.txt', 'github_projects.txt', 'github_projects.txt']
  Answer: The Paris Agreement climate classifier is a machine learning project completed for BSAN 6070. It used a Random Forest model selected after comparing multiple classifiers, achieving 0.933 ROC-AUC and 84.6% accuracy on the test set. The project involved feature engineering on climate policy data and model evaluation using cross-validation.

[Inference] How does this person's ML 

In [1]:
import pandas as pd

# Scores based on reviewing each answer above
scores = [
    {'retrieval_quality': 'Partial',  'faithfulness': 'Faithful', 'quality_score': 2},
    {'retrieval_quality': 'Partial',  'faithfulness': 'Faithful', 'quality_score': 3},
    {'retrieval_quality': 'Yes',      'faithfulness': 'Faithful', 'quality_score': 5},
    {'retrieval_quality': 'Yes',      'faithfulness': 'Partial',  'quality_score': 4},
    {'retrieval_quality': 'Yes',      'faithfulness': 'Partial',  'quality_score': 4},
    {'retrieval_quality': 'No',       'faithfulness': 'Faithful', 'quality_score': 5},
    {'retrieval_quality': 'Partial',  'faithfulness': 'Faithful', 'quality_score': 5},
    {'retrieval_quality': 'No',       'faithfulness': 'Faithful', 'quality_score': 5},
    {'retrieval_quality': 'Yes',      'faithfulness': 'Partial',  'quality_score': 4},
    {'retrieval_quality': 'Yes',      'faithfulness': 'Faithful', 'quality_score': 5},
]

eval_df = pd.DataFrame([
    {
        'Category': r['category'],
        'Question': r['question'][:55] + '...',
        'Sources': ', '.join(r['sources']),
        'Retrieval': s['retrieval_quality'],
        'Faithfulness': s['faithfulness'],
        'Score (1-5)': s['quality_score'],
    }
    for r, s in zip(results, scores)
])

print(f'Average quality score: {eval_df["Score (1-5)"].mean():.1f} / 5')
eval_df

Average quality score: 4.2 / 5


### Evaluation Analysis

**Where does the chatbot succeed?**
> Out-of-scope questions were handled correctly in all three cases — the chatbot refused to answer rather than hallucinate. Factual retrieval succeeded for well-scoped project questions (Q3 Paris Agreement, Q10 BSAN6070 model, both 5/5). Inference questions performed well (Q4, Q5 scored 4/5), showing the system can reason across documents. Zero hallucinations were recorded across all 10 questions.

**Where does it fail? Why?**
> The weakest results were Q1 (degree information, 2/5) and Q2 (programming languages, 3/5). Degree details are embedded in resume header sections that merge with other content during paragraph-aware chunking, reducing retrieval precision. Language coverage was incomplete because skills are spread across multiple document sections — at k=3, the CV chunks containing the full skills list ranked lower than Resume chunks for this query.

**What would you improve?**
> Increasing k from 3 to 5 would retrieve more chunks and likely surface the full skills list (Q2) and degree details (Q1). Adding a dedicated education and skills summary document to the data folder would also improve retrieval precision for basic profile questions without requiring changes to the retrieval logic.

---

## After Completing This Notebook

1. Copy `PROMPT_V3` → `streamlit_app.py` `SYSTEM_PROMPT`
2. Copy `chunk_paragraph` → `streamlit_app.py` `chunk_documents()`
3. Add 3 sample questions to `streamlit_app.py` `samples` list
4. Fill in your scores above, then update `evaluation/test_results.md`
5. Write `memo.md` and complete `ai_log.md` (8+ entries)

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option A*

### Top-k Sensitivity Analysis

`k` controls how many chunks are retrieved per query. More chunks means broader coverage but longer prompts and more potential for noise. The evaluation above used k=3. The two lowest-scoring questions both suffered from retrieval coverage gaps — degrees (Q1, 2/5) and programming languages (Q2, 3/5) are facts distributed across multiple document sections.

| k | Q1 "What degrees does this person hold?" | Q2 "What programming languages?" | Trade-off |
|---|---|---|---|
| k=3 | "I don't have that information" — degree header sections ranked below other content | Python + SQL retrieved; Tableau/SAS missed (in CV chunks that ranked 4th+) | Precise but incomplete for distributed facts |
| k=5 | More likely to surface education sections; degree chunks rank in top 5 | Full skills list surfaced across CV, Resume, and about.txt chunks | Better coverage; slightly longer prompt context |

**Decision:** k=3 is retained for the deployed app as a sensible default — it performs well on the majority of questions and keeps prompts concise. The retrieval gap for Q1 and Q2 is more effectively addressed by adding `data/about.txt`, which consolidates degree and skills information into a single chunk that ranks first for profile-oriented queries. This fix was implemented and is active in the deployed Streamlit app.